In [16]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import os

In [17]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

d:\_PROJECTS\_ACTIVE_LOCAL\OHCA_Prediction_PAROS\Mobile-AED-Study\src


In [18]:
BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

d:\_PROJECTS\_ACTIVE_LOCAL\OHCA_Prediction_PAROS\Mobile-AED-Study\datasets


In [19]:
read_from_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/ohca_data/ohca_binary_complete.csv")
final_model_df = pd.read_csv(read_from_filepath)
final_model_df = final_model_df.drop(columns=['Unnamed: 0'])

In [20]:
final_model_df.columns

Index(['pln_area_n', 'subzone_n', 'year', 'month', 'incident_count',
       'ohca_binary', 'total', 'above_60', 'above_60_proportion',
       'male_chinese_proportion', 'female_chinese_proportion',
       'male_malays_proportion', 'female_malays_proportion',
       'male_indians_proportion', 'female_indians_proportion',
       'male_others_proportion', 'female_others_proportion',
       'business_1_encoding', 'business_2_encoding', 'business_park_encoding',
       'school_encoding', 'airport', 'is_residential'],
      dtype='object')

In [21]:
predictor_cols = [
       'year', 'month',
       'above_60_proportion', 'male_chinese_proportion',
       'female_chinese_proportion', 'male_malays_proportion',
       'female_malays_proportion', 'male_indians_proportion',
       'female_indians_proportion', 'male_others_proportion',
       'female_others_proportion', 'business_1_encoding',
       'business_2_encoding', 'business_park_encoding', 'school_encoding',
       'airport', 'is_residential'
]

# Keep a reference dataframe for testing/comparison
comparison_df = final_model_df[['pln_area_n', 'subzone_n', 'year', 'month', 'ohca_binary']].copy()
comparison_df

,pln_area_n,subzone_n,year,month,ohca_binary
0,ang mo kio,ang mo kio town centre,2010,4,0
1,ang mo kio,ang mo kio town centre,2010,5,1
2,ang mo kio,ang mo kio town centre,2010,6,0
3,ang mo kio,ang mo kio town centre,2010,7,1
4,ang mo kio,ang mo kio town centre,2010,8,0
...,...,...,...,...,...
43564,western water catchment,murai,2021,8,0
43565,western water catchment,murai,2021,9,1
43566,western water catchment,murai,2021,10,1
43567,western water catchment,murai,2021,11,1


In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Chronological Split
# Training: 2010 - 2019
train_df = final_model_df[final_model_df['year'] <= 2019]
X_train = train_df[predictor_cols].fillna(0)
y_train = train_df['ohca_binary']

# Validation: 2020
val_df = final_model_df[final_model_df['year'] == 2020]
X_val = val_df[predictor_cols].fillna(0)
y_val = val_df['ohca_binary']

# Testing: 2021
test_df = final_model_df[final_model_df['year'] == 2021]
X_test = test_df[predictor_cols].fillna(0)
y_test = test_df['ohca_binary']

In [23]:
# Initialize and train the model
log_reg = LogisticRegression(max_iter=10000)
# log_reg = LogisticRegression(solver='saga', max_iter=5000)
log_reg.fit(X_train, y_train)

# 3. Validation Phase (2020 Data)
val_preds = log_reg.predict(X_val)
val_probs = log_reg.predict_proba(X_val)[:, 1]

print("--- Validation Results (2020) ---")
print(classification_report(y_val, val_preds))
print(f"AUC-ROC Score: {roc_auc_score(y_val, val_probs):.4f}")

# 4. Final Testing Phase (2021 Data)
test_preds = log_reg.predict(X_test)
test_probs = log_reg.predict_proba(X_test)[:, 1]
print("\n--- Final Test Results (2021) ---")
print(classification_report(y_test, test_preds))
print(f"AUC-ROC Score: {roc_auc_score(y_test, test_probs):.4f}")

--- Validation Results (2020) ---
              precision    recall  f1-score   support

           0       0.75      0.85      0.80      2169
           1       0.74      0.60      0.66      1539

    accuracy                           0.75      3708
   macro avg       0.74      0.73      0.73      3708
weighted avg       0.75      0.75      0.74      3708

AUC-ROC Score: 0.8205

--- Final Test Results (2021) ---
              precision    recall  f1-score   support

           0       0.74      0.85      0.79      2118
           1       0.75      0.60      0.66      1590

    accuracy                           0.74      3708
   macro avg       0.74      0.72      0.73      3708
weighted avg       0.74      0.74      0.74      3708

AUC-ROC Score: 0.8150


In [24]:
display(confusion_matrix(y_val, val_preds))
display(confusion_matrix(y_test, test_preds))

array([[1840,  329],
       [ 613,  926]], dtype=int64)

array([[1797,  321],
       [ 640,  950]], dtype=int64)

In [9]:
# Normalise proportion columns to 0 - 1 instead of 0 - 1000
# The idea is that the proportion columns have numbers 0 - 1000 
# which could potentially lower the model's accuracy 
# due to there being binary columns as well in the dataset
cols_to_fix = [c for c in final_model_df.columns if "proportion" in c]

for col in cols_to_fix:
    final_model_df[col] = final_model_df[col] / 1000

save_to_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/ohca_data/ohca_normalised_complete.csv")
final_model_df.to_csv(save_to_filepath)

train_df = final_model_df[final_model_df['year'] <= 2019]
X_train = train_df[predictor_cols].fillna(0)
y_train = train_df['ohca_binary']

# Validation: 2020
val_df = final_model_df[final_model_df['year'] == 2020]
X_val = val_df[predictor_cols].fillna(0)
y_val = val_df['ohca_binary']

# Testing: 2021
test_df = final_model_df[final_model_df['year'] == 2021]
X_test = test_df[predictor_cols].fillna(0)
y_test = test_df['ohca_binary']

In [10]:
# Initialize and train the model
log_reg = LogisticRegression(max_iter=5000)
# log_reg = LogisticRegression(solver='saga', max_iter=5000)
log_reg.fit(X_train, y_train)

# 3. Validation Phase (2020 Data)
val_preds = log_reg.predict(X_val)
val_probs = log_reg.predict_proba(X_val)[:, 1]

print("--- Validation Results (2020) ---")
print(classification_report(y_val, val_preds))
print(f"AUC-ROC Score: {roc_auc_score(y_val, val_probs):.4f}")

# 4. Final Testing Phase (2021 Data)
test_preds = log_reg.predict(X_test)
test_probs = log_reg.predict_proba(X_test)[:, 1]
print("\n--- Final Test Results (2021) ---")
print(classification_report(y_test, test_preds))
print(f"AUC-ROC Score: {roc_auc_score(y_test, test_probs):.4f}")

--- Validation Results (2020) ---
              precision    recall  f1-score   support

           0       0.75      0.86      0.81      2169
           1       0.76      0.60      0.67      1539

    accuracy                           0.76      3708
   macro avg       0.76      0.73      0.74      3708
weighted avg       0.76      0.76      0.75      3708

AUC-ROC Score: 0.8219

--- Final Test Results (2021) ---
              precision    recall  f1-score   support

           0       0.74      0.86      0.80      2118
           1       0.77      0.60      0.68      1590

    accuracy                           0.75      3708
   macro avg       0.76      0.73      0.74      3708
weighted avg       0.75      0.75      0.75      3708

AUC-ROC Score: 0.8150


/Users/yitong/opt/anaconda3/envs/qgis_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Normalising the proportion columns to 0 - 1 range seemed to not change the results much

In [11]:
display(confusion_matrix(y_val, val_preds))
display(confusion_matrix(y_test, test_preds))

array([[1874,  295],
       [ 609,  930]])

array([[1827,  291],
       [ 629,  961]])